In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, r2_score
from sklearn.ensemble import RandomForestRegressor

In [2]:
df=pd.read_csv("ml_features.csv")
df.head()

,assignment_id,utilization_pct,assignment_confirmed,speed_kmph,telemetry_lat,telemetry_lon,telemetry_recorded_at,anomaly_type,anomaly_severity,anomaly_detected_at,...,stop_sequence,shipment_id,origin_lat,origin_lon,dest_lat,dest_lon,shipment_weight_kg,shipment_priority,shipment_deadline,time_to_deadline_minutes
0,ASN_0015,100.0,f,31.3,16.140948,77.020445,2026-07-15 18:26:06+05:30,NaN,NaN,NaN,...,1,SHIP_1015,10.892186,77.031204,28.602120,77.006398,7555,MEDIUM,2026-07-21 10:45:17+05:30,8179.172549
1,ASN_0015,100.0,f,33.7,27.431302,77.003249,2026-07-17 03:18:25+05:30,NaN,NaN,NaN,...,2,SHIP_1015,10.892186,77.031204,28.602120,77.006398,7555,MEDIUM,2026-07-21 10:45:17+05:30,6206.859903
2,ASN_0015,100.0,f,56.9,28.519907,77.024451,2026-07-17 06:34:59+05:30,NaN,NaN,NaN,...,3,SHIP_1015,10.892186,77.031204,28.602120,77.006398,7555,MEDIUM,2026-07-21 10:45:17+05:30,6010.289060
3,ASN_0055,83.9,t,62.3,16.434473,78.817589,2026-07-17 09:55:02+05:30,NaN,NaN,NaN,...,1,SHIP_1055,17.378700,78.415710,13.133197,80.224386,2921,HIGH,2026-07-16 19:35:13+05:30,0.000000
4,ASN_0055,83.9,t,61.2,16.260334,78.904268,2026-07-17 10:23:19+05:30,NaN,NaN,NaN,...,2,SHIP_1055,17.378700,78.415710,13.133197,80.224386,2921,HIGH,2026-07-16 19:35:13+05:30,0.000000


In [3]:
#Tranforming Timestamp to Date-Time 
timestamp_cols = [
    "telemetry_recorded_at", "planned_arrival", "actual_arrival_at",
    "anomaly_detected_at", "shipment_deadline"
]
for col in timestamp_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

## Feature Derivation

In [4]:
#Creating Label (duration left) and converting into minutes
df["remaining_duration_minutes"] = (df["actual_arrival_at"] - df["telemetry_recorded_at"]).dt.total_seconds() / 60

In [5]:
df = df[df["remaining_duration_minutes"] > 0]  # drop invalid/negative labels
df = df[df["remaining_duration_minutes"].notna()] # drop trips that haven't got completed. We will test our model on these.

### Calculating Remaining Distance

In [6]:
from geopy.distance import geodesic

def compute_distance(row):
    origin = (row["telemetry_lat"], row["telemetry_lon"])
    dest = (row["dest_lat"], row["dest_lon"])
    return geodesic(origin, dest).km

df["distance_remaining_km"] = df.apply(compute_distance, axis=1)

In [7]:
#Extracting Time & Day of week from Timestamp - Encoding needed if we weren't using XGBoost / LightGBM / Random Forest
df["time_of_day_hr"] = df["telemetry_recorded_at"].dt.hour + df["telemetry_recorded_at"].dt.minute / 60
df["day_of_week"] = df["telemetry_recorded_at"].dt.dayofweek  # 0=Monday ... 6=Sunday

In [8]:
#Stops remaining
df = df.sort_values(["shipment_id", "telemetry_recorded_at"])
df["stops_remaining"] = df.groupby("shipment_id")["stop_sequence"].transform("max") - df["stop_sequence"]

In [9]:
print("shipment_id" in df.columns)  # should be True here
print(df[["shipment_id", "stop_sequence", "stops_remaining"]].head())

True
    shipment_id  stop_sequence  stops_remaining
558   SHIP_1001              1                3
559   SHIP_1001              2                2
560   SHIP_1001              3                1
561   SHIP_1001              4                0
684   SHIP_1002              1                5


In [10]:
#Anomaly Recency
df["anomaly_recency_minutes"] = (df["telemetry_recorded_at"] - df["anomaly_detected_at"]).dt.total_seconds() / 60

# Rows with no anomaly (anomaly_detected_at is NaT) will naturally become NaN here.
# Fill with a sentinel value that clearly signals "no anomaly" -- NOT 0.
df["anomaly_recency_minutes"] = df["anomaly_recency_minutes"].fillna(-1)

In [11]:
print(df[[
    "remaining_duration_minutes", "distance_remaining_km",
    "time_of_day_hr", "day_of_week", "stops_remaining", "anomaly_recency_minutes"
]].describe())

print(df.isna().sum())

       remaining_duration_minutes  distance_remaining_km  time_of_day_hr  \
count                  942.000000             942.000000      942.000000   
mean                   636.918064             453.723172       11.918560   
std                    504.389603             359.663291        6.671939   
min                      1.183333               0.350778        0.033333   
25%                    236.237500             165.686587        6.366667   
50%                    500.350000             356.045238       11.658333   
75%                    946.600000             681.496528       17.679167   
max                   2699.050000            1925.059720       23.916667   

       day_of_week  stops_remaining  anomaly_recency_minutes  
count   942.000000       942.000000               942.000000  
mean      3.574310         2.069002                -3.770382  
std       1.471767         1.670687               281.838431  
min       0.000000         0.000000             -1545.616667  


## Data Preprocess

In [12]:
# Replace missing string/categorical values from anomaly fields
df['anomaly_type'] = df['anomaly_type'].fillna('None')
df['anomaly_severity'] = df['anomaly_severity'].fillna(0)

In [13]:
columns_to_drop = [
    # Identifiers - unique per row, no predictive pattern for a tree to learn
    "assignment_id", "vehicle_id", "plate_number", "shipment_id",

    # Unstructured - not usable in raw form
    "waypoints_json",

    # Raw timestamps - already converted into numeric features (Steps above);
    # keeping the raw datetime columns serves no purpose for the model
    "telemetry_recorded_at", "planned_arrival",
    "anomaly_detected_at", "shipment_deadline",

    # CRITICAL: drop actual_arrival_at entirely.
    # This is literally the value used to compute your label (remaining_duration_minutes). Leaving it in the feature set is direct label leakage -- the model would "cheat."
    "actual_arrival_at",]

In [14]:
df = df.drop(columns=columns_to_drop)

In [15]:
df=df.drop("vehicle_current_lat",axis=1)
df=df.drop("vehicle_current_lon",axis=1)

In [16]:
print(df.isna().sum())

utilization_pct               0
assignment_confirmed          0
speed_kmph                    0
telemetry_lat                 0
telemetry_lon                 0
anomaly_type                  0
anomaly_severity              0
driver_status                 0
vehicle_capacity_kg           0
total_distance_km             0
stop_sequence                 0
origin_lat                    0
origin_lon                    0
dest_lat                      0
dest_lon                      0
shipment_weight_kg            0
shipment_priority             0
time_to_deadline_minutes      0
remaining_duration_minutes    0
distance_remaining_km         0
time_of_day_hr                0
day_of_week                   0
stops_remaining               0
anomaly_recency_minutes       0
dtype: int64


In [17]:
#Converting Assignment confirmed to Boolean
df["assignment_confirmed"] = df["assignment_confirmed"].map({"t": True, "f": False})

In [18]:
#Label Concoding
categorical_cols = ["anomaly_type", "driver_status", "shipment_priority"]
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le   

In [19]:
df.head()

,utilization_pct,assignment_confirmed,speed_kmph,telemetry_lat,telemetry_lon,anomaly_type,anomaly_severity,driver_status,vehicle_capacity_kg,total_distance_km,...,dest_lon,shipment_weight_kg,shipment_priority,time_to_deadline_minutes,remaining_duration_minutes,distance_remaining_km,time_of_day_hr,day_of_week,stops_remaining,anomaly_recency_minutes
558,100.0,False,62.7,15.250602,79.224573,3,0.0,2,1500,590.7,...,78.339864,2631,2,5791.358207,286.383333,244.491363,6.166667,2,3,-1.0
559,100.0,False,52.7,16.066017,78.887668,3,0.0,2,1500,590.7,...,78.339864,2631,2,5681.961026,176.983333,147.318254,8.000000,2,2,-1.0
560,100.0,False,60.6,16.066783,78.869629,3,0.0,2,1500,590.7,...,78.339864,2631,2,5672.608777,167.633333,146.487485,8.150000,2,1,-1.0
561,100.0,False,47.9,16.940596,78.517684,3,0.0,2,1500,590.7,...,78.339864,2631,2,5557.304457,52.333333,42.855058,10.066667,2,0,-1.0
684,100.0,False,53.4,23.635401,81.308659,3,0.0,0,1500,1768.9,...,85.290257,7579,2,5750.376790,587.916667,458.017580,23.100000,2,5,-1.0


## Model Training/Testing

In [20]:
X=df.drop("remaining_duration_minutes",axis=1)
y=df["remaining_duration_minutes"]

In [21]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [22]:
rf=RandomForestRegressor(n_estimators=501,max_features="sqrt",oob_score=True,bootstrap=True)
rf.fit(X_train,y_train)

y_pred_test=rf.predict(X_test)
y_pred_train=rf.predict(X_train)

print("Training R2: ",r2_score(y_train,y_pred_train)*100,"%")
print("Testing R2: ",r2_score(y_test,y_pred_test)*100,"%")

Training R2:  99.22392326463371 %
Testing R2:  95.3180212577212 %


## Taking Input from User

In [23]:
import joblib

joblib.dump(rf, "eta_model.pkl")
joblib.dump(encoders, "encoders.pkl")

# Also save the exact column order the model was trained on -- critical for prediction
joblib.dump(list(X_train.columns), "feature_columns.pkl")

['feature_columns.pkl']